# 01 â€” QuickDraw Data Exploration

Downloads a small sample from 5 QuickDraw categories, converts bitmaps to images,
visualises sample grids, plots class distributions, and computes per-channel
mean/std for normalisation.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import requests
from PIL import Image
import io, os, sys

sys.path.insert(0, os.path.join('..', 'src'))
print('Imports OK')

Imports OK


c:\Users\jayan\AppData\Local\Programs\Python\Python311\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.0)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


## 1. Download samples via the QuickDraw numpy_bitmap API

In [3]:
BASE_URL   = 'https://storage.googleapis.com/quickdraw_dataset/full/numpy_bitmap/'
CATEGORIES = ['face', 'house', 'tree', 'cat', 'car', 'bird', 'dog', 'fish', 'flower', 'bicycle']
N_SAMPLES  = 5000   # per category (full training scale)
SAVE_DIR   = os.path.join('..', 'data', 'raw', 'quickdraw')
os.makedirs(SAVE_DIR, exist_ok=True)

data = {}   # category -> ndarray (N, 784) uint8

for cat in CATEGORIES:
    fname = os.path.join(SAVE_DIR, f'{cat}.npy')

    # Re-download if file exists but has fewer samples than needed
    if os.path.exists(fname):
        existing = np.load(fname)
        if len(existing) >= N_SAMPLES:
            print(f'  [{cat}] already have {len(existing)} samples - skipping.')
            data[cat] = existing[:N_SAMPLES]
            continue
        else:
            print(f'  [{cat}] only {len(existing)} samples - re-downloading at full scale.')

    url = BASE_URL + cat.replace(' ', '%20') + '.npy'
    print(f'  Fetching {url} ...')
    r = requests.get(url, stream=True)
    r.raise_for_status()

    raw = np.load(io.BytesIO(r.content))   # shape (total_in_dataset, 784)
    data[cat] = raw[:N_SAMPLES]
    np.save(fname, data[cat])
    print(f'    saved {data[cat].shape}  ({os.path.getsize(fname)/1e6:.1f} MB)')

print(f'Download complete. {sum(len(v) for v in data.values())} total drawings.')

  [face] already have 5000 samples - skipping.
  [house] already have 5000 samples - skipping.
  [tree] already have 5000 samples - skipping.
  [cat] already have 5000 samples - skipping.
  [car] already have 5000 samples - skipping.
  Fetching https://storage.googleapis.com/quickdraw_dataset/full/numpy_bitmap/bird.npy ...
    saved (5000, 784)  (3.9 MB)
  [dog] only 500 samples - re-downloading at full scale.
  Fetching https://storage.googleapis.com/quickdraw_dataset/full/numpy_bitmap/dog.npy ...
    saved (5000, 784)  (3.9 MB)
  Fetching https://storage.googleapis.com/quickdraw_dataset/full/numpy_bitmap/fish.npy ...
    saved (5000, 784)  (3.9 MB)
  Fetching https://storage.googleapis.com/quickdraw_dataset/full/numpy_bitmap/flower.npy ...
    saved (5000, 784)  (3.9 MB)
  [bicycle] only 500 samples - re-downloading at full scale.
  Fetching https://storage.googleapis.com/quickdraw_dataset/full/numpy_bitmap/bicycle.npy ...
    saved (5000, 784)  (3.9 MB)
Download complete. 50000 tota

## 2. Convert numpy bitmaps to PIL images

In [ ]:
from dataset import numpy_bitmap_to_image

images = {}  # category -> list of PIL images
for cat in CATEGORIES:
    images[cat] = [numpy_bitmap_to_image(row, image_size=224) for row in data[cat]]

print('Conversion done. Sample size:', images[CATEGORIES[0]][0].size)

## 3. Visualise a 5Ã—5 grid per category

In [ ]:
import random

for cat in CATEGORIES:
    sample = random.sample(images[cat], 25)
    fig, axes = plt.subplots(5, 5, figsize=(10, 10))
    fig.suptitle(f'Category: {cat}', fontsize=14)
    for ax, img in zip(axes.flatten(), sample):
        ax.imshow(img)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

## 4. Class distribution

In [ ]:
counts = {cat: len(imgs) for cat, imgs in images.items()}

plt.figure(figsize=(8, 4))
plt.bar(counts.keys(), counts.values(), color='steelblue', edgecolor='black')
plt.title('Class Distribution â€” QuickDraw Sample')
plt.xlabel('Category')
plt.ylabel('Number of Drawings')
plt.tight_layout()
plt.show()

print('Counts:', counts)

## 5. Image dimensions, pixel value ranges, data types

In [ ]:
sample_img = images[CATEGORIES[0]][0]
arr = np.array(sample_img)

print(f'PIL mode  : {sample_img.mode}')
print(f'Size      : {sample_img.size}  (WÃ—H)')
print(f'Array shape : {arr.shape}')
print(f'Dtype     : {arr.dtype}')
print(f'Min pixel : {arr.min()}')
print(f'Max pixel : {arr.max()}')

## 6. Per-channel mean and std for normalisation

In [ ]:
all_arrays = []
for cat in CATEGORIES:
    for img in images[cat]:
        all_arrays.append(np.array(img).astype(np.float32) / 255.0)

stack = np.stack(all_arrays, axis=0)   # (N, H, W, 3)
mean = stack.mean(axis=(0, 1, 2))      # per-channel
std  = stack.std(axis=(0, 1, 2))

print('Per-channel mean (R, G, B):', mean.round(4))
print('Per-channel std  (R, G, B):', std.round(4))
print()
print('(These values are for the QuickDraw-specific subset.')
print(' The model uses ImageNet defaults: mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])')